# 7. Tổng lũy kế
Dự đoán mỗi dòng kết quả biểu diễn điều gì.

In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'run_inventory.py').is_file())
sys.path.insert(0, str(root))
from inventory_lab.db import connect, DEFAULT_DB, LAST
from inventory_lab.sql_metrics import query
db = Path(os.environ.get('INVENTORY_LAB_DB', DEFAULT_DB))
assert db.is_file(), 'Chạy python src/prepare_data.py trong PowerShell'
def read(sql, params=None):
    with connect(db) as connection:
        return pd.read_sql_query(sql, connection, params=params)

result = read("WITH lots(id,stock) AS (VALUES ('A',8),('B',10)), ranked AS (SELECT id, stock, COALESCE(SUM(stock) OVER(ORDER BY id ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING),0) AS prior FROM lots) SELECT id,MAX(0,MIN(stock,12-prior)) AS allocated FROM ranked ORDER BY id")
result.head()

Khung cửa sổ loại dòng hiện tại để tính lượng đã có thể lấy từ những lô trước.

**Tự làm**

Thử yêu cầu 8 và 21 hộp.

In [ ]:
assert result.allocated.tolist() == [8,4]